# report

> `ComparisonResultList` -- the list-with-reporting-helpers that `compare()` returns.

`compare()` (built in `03_compare`, the next notebook) gives us back a list
of `ComparisonResult` objects, one per engine. A plain list can already do
the basics -- we can loop over it, index into it, check its length -- but on
its own it can't answer the two questions we actually care about once a
comparison finishes: "put this side by side for me to read" and "show me
exactly what differs between two of these engines." `ComparisonResultList`
is a list that also knows how to answer those two questions, so we never
have to write that formatting code ourselves each time we run a comparison.

In [ ]:
#| default_exp report

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
from __future__ import annotations

import json
from pathlib import Path

from estravon_bench.io import ComparisonResult

## `ComparisonResultList`

We define this as a subclass of Python's built-in `list` rather than a
separate wrapper object holding a list inside it. That choice is deliberate:
it means `compare()` can hand us back something that behaves exactly like a
normal list everywhere we'd expect one to (`for r in results:`,
`results[0]`, `len(results)`, `list(results)`) while also carrying the three
extra methods below. We never have to "unwrap" it to get at our results.

- **`to_markdown_table()`** turns the whole list into one side-by-side table
  -- engine, time, cost, whether it ran locally, page count, and status.
  This is the everyday output of a comparison: read down the columns and we
  can already see which engine was fastest, which was free, and which
  failed, without opening anything else. We love markdown.
- **`get(engine)`** finds one specific engine's result by name, or gives us
  `None` if that engine isn't in the list -- a small convenience so we don't
  have to write our own search loop when we want to look at just one
  engine's output in detail.
- **`diff(engine_a, engine_b)`** shows us, line by line, exactly where two
  engines' Markdown output disagrees -- the same format `git diff` or
  `diff -u` produce. A table tells us *that* two engines differ in timing or
  cost; `diff()` is how we actually see *what* one engine wrote differently
  from another on the same page.

In [ ]:
#| export
class ComparisonResultList(list):
    """list[ComparisonResult] with side-by-side reporting helpers.

    compare() returns this type directly -- callers can treat it as a plain
    list (iterate, index, len()) or call the extra methods below.
    """

    def to_markdown_table(self) -> str:
        """Side-by-side engine | time | cost | local? | pages | images | status table."""
        header = "| engine | time (s) | cost (usd) | local | pages | images | status |\n"
        header += "|---|---|---|---|---|---|---|\n"
        rows = []
        for r in self:
            if not r.ok:
                rows.append(f"| {r.engine} | - | - | - | - | - | error: {r.error} |")
                continue
            time_s = f"{r.predict_time_s:.2f}" if r.predict_time_s is not None else "-"
            cost = "free (local)" if r.local else (f"${r.cost_usd:.4f}" if r.cost_usd is not None else "-")
            pages = r.page_count if r.page_count is not None else "-"
            rows.append(f"| {r.engine} | {time_s} | {cost} | {'yes' if r.local else 'no'} | {pages} | {len(r.images)} | ok |")
        return header + "\n".join(rows)

    def get(self, engine: str) -> ComparisonResult | None:
        for r in self:
            if r.engine == engine:
                return r
        return None

    def diff(self, engine_a: str, engine_b: str) -> str:
        """Unified line diff of two engines' Markdown output, for eyeballing."""
        import difflib
        a, b = self.get(engine_a), self.get(engine_b)
        if a is None or b is None:
            missing = engine_a if a is None else engine_b
            raise KeyError(f"no result for engine {missing!r}")
        if not a.ok or not b.ok:
            raise ValueError("cannot diff an engine that errored")
        a_lines = (a.markdown or "").splitlines(keepends=True)
        b_lines = (b.markdown or "").splitlines(keepends=True)
        return "".join(difflib.unified_diff(a_lines, b_lines, fromfile=engine_a, tofile=engine_b))

    def __add__(self, other):
        """`a + b` on two ComparisonResultLists (or a plain list) stays a
        ComparisonResultList -- list's own __add__ would silently downgrade
        the result to a plain list, losing every method above with no error
        until first use. Neither operand is mutated."""
        return ComparisonResultList(list(self) + list(other))

    def __radd__(self, other):
        """Handles `plain_list + comparison_result_list` -- Python tries the
        right operand's __radd__ first when its type is a subclass of the
        left operand's, which a plain list is not aware of on its own."""
        return ComparisonResultList(list(other) + list(self))

    @classmethod
    def merge(cls, *lists: "list[ComparisonResult]") -> "ComparisonResultList":
        """Combine any number of lists -- e.g. results from separate
        one-engine-at-a-time compare() calls -- into one new
        ComparisonResultList. None of the inputs are mutated. Equivalent to
        chaining `+`, but reads more clearly for more than two lists and
        doesn't require importing itertools yourself."""
        import itertools
        return cls(itertools.chain.from_iterable(lists))

    def save(self, dir_path: str) -> Path:
        """Save this list to dir_path/ as manifest.json plus one
        subdirectory per engine (result.md + images/) -- plain files, no new
        dependency, so any file browser/git/image viewer can inspect a saved
        run directly. Pass a fresh dir_path per run (e.g. a timestamp) rather
        than reusing one across runs -- this does not merge into an existing
        save."""
        dir_path = Path(dir_path)
        dir_path.mkdir(parents=True, exist_ok=True)
        manifest = {"results": []}
        for r in self:
            engine_dir = dir_path / r.engine
            entry = {
                "engine": r.engine,
                "predict_time_s": r.predict_time_s,
                "cost_usd": r.cost_usd,
                "local": r.local,
                "page_count": r.page_count,
                "backend_url": r.backend_url,
                "error": r.error,
                "markdown_path": None,
                "image_paths": {},
            }
            if r.markdown is not None:
                engine_dir.mkdir(parents=True, exist_ok=True)
                md_path = engine_dir / "result.md"
                md_path.write_text(r.markdown)
                entry["markdown_path"] = str(md_path.relative_to(dir_path))
            if r.images:
                images_dir = engine_dir / "images"
                images_dir.mkdir(parents=True, exist_ok=True)
                for filename, data in r.images.items():
                    (images_dir / filename).write_bytes(data)
                    entry["image_paths"][filename] = str((images_dir / filename).relative_to(dir_path))
            manifest["results"].append(entry)
        (dir_path / "manifest.json").write_text(json.dumps(manifest, indent=2))
        return dir_path

    @classmethod
    def load(cls, dir_path: str) -> "ComparisonResultList":
        """Reload a list previously written by save() -- reads manifest.json
        plus each referenced result.md/image file back into real
        ComparisonResult objects."""
        dir_path = Path(dir_path)
        manifest = json.loads((dir_path / "manifest.json").read_text())
        out = cls()
        for entry in manifest["results"]:
            markdown = None
            if entry["markdown_path"] is not None:
                markdown = (dir_path / entry["markdown_path"]).read_text()
            images = {
                filename: (dir_path / rel_path).read_bytes()
                for filename, rel_path in entry["image_paths"].items()
            }
            out.append(ComparisonResult(
                engine=entry["engine"], markdown=markdown, images=images,
                predict_time_s=entry["predict_time_s"], cost_usd=entry["cost_usd"],
                local=entry["local"], page_count=entry["page_count"],
                backend_url=entry["backend_url"], error=entry["error"],
            ))
        return out

### Try it

We build a small `ComparisonResultList` by hand below -- one engine that
succeeded (`mineru`, free and local) and one that failed (`mistral`, no API
key) -- the same shape `compare()` would hand us after a real run. Printing
`to_markdown_table()` shows us the successful row and the failed row
rendered side by side, including the "free (local)" cost label rather than
a bare `0.0`. Then we deliberately try to `diff()` the failed engine against
the working one, to see that it refuses with a clear error instead of
crashing on missing Markdown -- there's nothing sensible to diff when one
side never produced any text.

In [ ]:
#| hide
lst = ComparisonResultList([
    ComparisonResult(engine="mineru", markdown="# Hi\nfoo", cost_usd=0.0, local=True, page_count=3, predict_time_s=12.0,
                      images={"img_001.jpg": b"\xff\xd8\xfffakejpeg"}),
    ComparisonResult(engine="mistral", error="no API key"),
])
table = lst.to_markdown_table()
assert "mineru" in table and "free (local)" in table
assert "error: no API key" in table
assert lst.get("mineru").page_count == 3
assert lst.get("nope") is None
print(table)
try:
    lst.diff("mineru", "mistral")
    raise AssertionError("should have raised")
except ValueError as exc:
    print(f"diff() correctly raised: {exc}")

### Try it: combining lists, and saving/loading a run

Three things worth confirming for real, not just from reading the code:
`+`/`merge()` correctly preserve `ComparisonResultList` (plain list's own `__add__` would silently drop every method above), and `save()`/`load()` round-trip an actual run through disk -- markdown, images, and metadata all still matching afterwards.


In [ ]:
#| hide
import tempfile as _tf

a = ComparisonResultList([ComparisonResult(engine="mineru", markdown="# A", local=True)])
b = ComparisonResultList([ComparisonResult(engine="mistral", markdown="# B", cost_usd=0.01)])
plain = [ComparisonResult(engine="replicate", markdown="# C")]

combined = a + b
assert type(combined) is ComparisonResultList
assert [r.engine for r in combined] == ["mineru", "mistral"]

combined2 = plain + a   # plain list on the LEFT -- the case __radd__ exists for
assert type(combined2) is ComparisonResultList
assert [r.engine for r in combined2] == ["replicate", "mineru"]

merged = ComparisonResultList.merge(a, b, plain)
assert type(merged) is ComparisonResultList
assert [r.engine for r in merged] == ["mineru", "mistral", "replicate"]
assert len(a) == 1   # merge() does not mutate its inputs
print("combine ops OK:", [r.engine for r in merged])

# save() / load() round trip -- a real success row with images, and a real error row
to_save = ComparisonResultList([
    ComparisonResult(engine="mineru", markdown="# Hi\nfoo", cost_usd=0.0, local=True,
                      page_count=2, predict_time_s=12.0, backend_url="http://127.0.0.1:7860",
                      images={"img_001.jpg": b"\xff\xd8\xfffakejpeg", "img_002.png": b"\x89PNGfake"}),
    ComparisonResult(engine="mistral", error="no API key"),
])
with _tf.TemporaryDirectory() as _d:
    saved_path = to_save.save(f"{_d}/run")
    reloaded = ComparisonResultList.load(saved_path)

assert type(reloaded) is ComparisonResultList
assert len(reloaded) == 2
r0 = reloaded.get("mineru")
assert r0.markdown == "# Hi\nfoo"
assert r0.images == {"img_001.jpg": b"\xff\xd8\xfffakejpeg", "img_002.png": b"\x89PNGfake"}
assert r0.cost_usd == 0.0 and r0.local is True and r0.page_count == 2
r1 = reloaded.get("mistral")
assert not r1.ok and r1.error == "no API key" and r1.markdown is None and r1.images == {}
print("save()/load() round-trip OK:", reloaded.to_markdown_table())


---
Next: [`03_compare`](03_compare.ipynb) -- `compare()` itself, which builds
the `ComparisonResultList` we've just been exploring, using the `Client`
and `LocalEngineProcess` from `01_client`.

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()